In [1]:
from hexarena import STORE_DIR
from jarvis.utils import tqdm
from irc.manager import LikelihoodManager
from hexarena.scripts.common import create_env

env = create_env(gamma=1, no_arena=True)
manager = LikelihoodManager(env, STORE_DIR/'likelihoods')
policies = manager.policies

# Update configs with 'native' field

In [2]:
keys = list(policies.configs.keys())
policies.configs.cache = None
updated = set()
for key in keys:
    config = policies.configs[key]
    if 'native' not in config:
        config.native = False
        policies.configs[key] = config
        updated.add(key)
policies.configs.cache = {}
print(f'{len(updated)} configs updated')

0 configs updated


# Update optimality in regress_stats

In [3]:
keys = list(policies.ckpts.keys())
updated = set()
for key in keys:
    ckpt = policies.ckpts[key]
    if 'regress_stats' not in ckpt:
        continue
    for label in ['update', 'init']:
        stats = ckpt['regress_stats'][label]
        if stats['optimality']<=1:
            losses = (stats['losses_train']*stats['alphas']).sum(axis=1)
            queries, fits, optimality, fvu = summarize_progress(losses, ascending=False)
            stats['optimality'] = optimality
            updated.add(key)
    policies.ckpts[key] = ckpt
print(f'{len(updated)} policies updated')

0 policies updated


# Update policy stats with policy training progress

In [10]:
keys = list(policies.ckpts.keys())
updated = set()
for key in tqdm(keys):
    stat = policies.stats[key]
    policies.setup(policies.configs[key])
    policies.load_ckpt(policies.ckpts[key])
    stat.update(policies.get_stat())
    policies.stats[key] = stat
    updated.add(key)
print(f'{len(updated)} stats updated')

  0%|                                                                                                         …

1450 stats updated
